# 🎶 Spotify Hit Predictor

**Notebook 2 — Limpieza, consolidación y feature engineering**

**Rol responsable:** Data Engineer  



In [1]:
%pip install pandas numpy
import pandas as pd
import numpy as np

Note: you may need to restart the kernel to use updated packages.


In [2]:
df_raw = pd.read_csv("../data/raw/spotify-tracks-dataset.csv")

print("Shape original:", df_raw.shape)
df_raw.head()

Shape original: (114000, 22)


,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
df_clean = df_raw.copy()

# Eliminar columnas de índice generadas por exportaciones previas
columns_to_drop = ["Unnamed: 0.1", "Unnamed: 0"]

df_clean = df_clean.drop(columns=columns_to_drop)

print("Shape después de quitar índices:", df_clean.shape)
df_clean.head()

Shape después de quitar índices: (114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
# Eliminar registro incompleto con duración inválida

invalid_record_mask = (
    df_clean["artists"].isna()
    & df_clean["album_name"].isna()
    & df_clean["track_name"].isna()
    & (df_clean["duration_ms"] == 0)
)

print("Registros inválidos encontrados:", invalid_record_mask.sum())

df_clean = df_clean.loc[~invalid_record_mask].copy()

print("Shape después de eliminar el registro inválido:", df_clean.shape)

Registros inválidos encontrados: 1
Shape después de eliminar el registro inválido: (113999, 20)


In [5]:
# Verificar si una misma canción cambia en columnas distintas al género

columns_to_check = [
    "artists",
    "album_name",
    "track_name",
    "popularity",
    "duration_ms",
    "explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"
]

variation_by_track = (
    df_clean.groupby("track_id")[columns_to_check]
    .nunique(dropna=False)
)

tracks_with_conflicts = variation_by_track[
    (variation_by_track > 1).any(axis=1)
]

print("Canciones con diferencias fuera de track_genre:", len(tracks_with_conflicts))

tracks_with_conflicts.head()

Canciones con diferencias fuera de track_genre: 720


,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
track_id,,,,,,,,,,,,,,,,,,
00YwP3wJWiG8IxAA7OS9lo,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1
014SIjoLDG1Ku19c5FlDYh,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1
02jLfqc9gMo8PkHEGHY3OT,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1
03mHinvLdrdSTd7w4GPXwH,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1
04IUJPcxiUuXih8eJIVzGz,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [6]:
# Revisar un ejemplo de canción con conflicto de popularidad

example_track_id = tracks_with_conflicts.index[0]

df_clean.loc[
    df_clean["track_id"] == example_track_id,
    [
        "track_id",
        "artists",
        "album_name",
        "track_name",
        "popularity",
        "duration_ms",
        "track_genre"
    ]
].sort_values("popularity")

,track_id,artists,album_name,track_name,popularity,duration_ms,track_genre
99268,00YwP3wJWiG8IxAA7OS9lo,Anupam Roy,Doorbiney Chokh Rakhbona,Amake Amar Moto Thakte Dao,46,319946,singer-songwriter
102269,00YwP3wJWiG8IxAA7OS9lo,Anupam Roy,Doorbiney Chokh Rakhbona,Amake Amar Moto Thakte Dao,46,319946,songwriter
55776,00YwP3wJWiG8IxAA7OS9lo,Anupam Roy,Doorbiney Chokh Rakhbona,Amake Amar Moto Thakte Dao,47,319946,indian
56324,00YwP3wJWiG8IxAA7OS9lo,Anupam Roy,Doorbiney Chokh Rakhbona,Amake Amar Moto Thakte Dao,47,319946,indie-pop
57529,00YwP3wJWiG8IxAA7OS9lo,Anupam Roy,Doorbiney Chokh Rakhbona,Amake Amar Moto Thakte Dao,47,319946,indie
65813,00YwP3wJWiG8IxAA7OS9lo,Anupam Roy,Doorbiney Chokh Rakhbona,Amake Amar Moto Thakte Dao,47,319946,k-pop


In [7]:
# Revisar si las variaciones de popularidad cambian la clase popular/no popular

popularity_range_by_track = (
    df_clean.groupby("track_id")["popularity"]
    .agg(["min", "max", "count"])
)

conflicts_crossing_threshold = popularity_range_by_track[
    (popularity_range_by_track["min"] < 50)
    & (popularity_range_by_track["max"] >= 50)
]

print("Canciones con popularidad variable:", (popularity_range_by_track["min"] != popularity_range_by_track["max"]).sum())
print("Canciones cuya variación cruza el umbral 50:", len(conflicts_crossing_threshold))

conflicts_crossing_threshold.head(10)

Canciones con popularidad variable: 720
Canciones cuya variación cruza el umbral 50: 10


,min,max,count
track_id,,,
05ztmc9JMq8pYE2GyY72Zd,49,50,2
0qJOO0qpBFmR1vXf4rMRuj,49,50,2
13W4Zc0dy3tzOzlKyuIsMh,49,50,3
1D18MMOF1hNEurbJPv0QCH,49,50,2
1vNqZdnSaRIMAoVW9oJR2d,49,50,2
2CfNnviw8KsMGGcJ5RtZ61,49,50,3
4isHeZRxdYw7IgGvl8OPwq,49,50,2
4tQeEHoii36QEGOEacAybz,49,50,3
5453frA2hsBPjGRVDHm1PJ,49,50,2


In [8]:
# Identificar canciones con popularidad ambigua alrededor del umbral

threshold = 50

ambiguous_track_ids = conflicts_crossing_threshold.index.tolist()

print("Canciones con etiqueta ambigua:", len(ambiguous_track_ids))

df_clean.loc[
    df_clean["track_id"].isin(ambiguous_track_ids),
    [
        "track_id",
        "artists",
        "track_name",
        "popularity",
        "track_genre"
    ]
].sort_values(["track_id", "popularity"])

Canciones con etiqueta ambigua: 10


,track_id,artists,track_name,popularity,track_genre
94545,05ztmc9JMq8pYE2GyY72Zd,Ouse;Powfu,Too Many Problems,49,sad
33436,05ztmc9JMq8pYE2GyY72Zd,Ouse;Powfu,Too Many Problems,50,emo
94476,0qJOO0qpBFmR1vXf4rMRuj,Promoting Sounds;Ouse;Kam Michael,i still think of u,49,sad
33319,0qJOO0qpBFmR1vXf4rMRuj,Promoting Sounds;Ouse;Kam Michael,i still think of u,50,emo
97427,13W4Zc0dy3tzOzlKyuIsMh,Almir Sater;Renato Teixeira,A Flor Que A Gente Assopra,49,sertanejo
34395,13W4Zc0dy3tzOzlKyuIsMh,Almir Sater;Renato Teixeira,A Flor Que A Gente Assopra,50,folk
74034,13W4Zc0dy3tzOzlKyuIsMh,Almir Sater;Renato Teixeira,A Flor Que A Gente Assopra,50,mpb
97306,1D18MMOF1hNEurbJPv0QCH,Thiago Brava;Unha Pintada,Alô Polícia,49,sertanejo
37416,1D18MMOF1hNEurbJPv0QCH,Thiago Brava;Unha Pintada,Alô Polícia,50,funk
94470,1vNqZdnSaRIMAoVW9oJR2d,yaeow,I'm Just a Ghost,49,sad


In [9]:
# Crear dataset con una fila por canción y todos sus géneros asociados

# 1. Separar canciones ambiguas para no usarlas en entrenamiento
df_non_ambiguous = df_clean.loc[
    ~df_clean["track_id"].isin(ambiguous_track_ids)
].copy()

print("Filas antes de agrupar:", df_non_ambiguous.shape[0])

# 2. Columnas que deben mantenerse como una sola observación por track_id
single_value_columns = [
    "artists",
    "album_name",
    "track_name",
    "duration_ms",
    "explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"
]

# 3. Crear una fila por canción
df_model = (
    df_non_ambiguous
    .groupby("track_id", as_index=False)
    .agg(
        {
            **{col: "first" for col in single_value_columns},
            "popularity": "median",
            "track_genre": lambda x: sorted(set(x))
        }
    )
)

# 4. Renombrar para dejar claro que ahora contiene varios géneros
df_model = df_model.rename(columns={"track_genre": "genres"})

print("Shape después de agrupar por canción:", df_model.shape)

df_model[
    [
        "track_id",
        "artists",
        "track_name",
        "popularity",
        "genres"
    ]
].head()

Filas antes de agrupar: 113976
Shape después de agrupar por canción: (89730, 20)


,track_id,artists,track_name,popularity,genres
0,0000vdREvCVMxbQTkS888c,Rill,Lolly,44.0,[german]
1,000CC8EParg64OmTxVnZ0p,Glee Cast,It's All Coming Back To Me Now (Glee Cast Vers...,47.0,[club]
2,000Iz0K615UepwSJ5z2RE5,Paul Kalkbrenner;Pig&Dan,Böxig Leise - Pig & Dan Remix,22.0,[minimal-techno]
3,000RDCYioLteXcutOjeweY,Jordan Sandhu,Teeje Week,62.0,[hip-hop]
4,000qpdoc97IMTBvF8gwcpy,Paul Kalkbrenner,Tief,19.0,[minimal-techno]


In [10]:
# Crear variable objetivo de popularidad

threshold = 50

df_model["popular"] = (
    df_model["popularity"] >= threshold
).astype(int)

target_summary = pd.DataFrame({
    "cantidad": df_model["popular"].value_counts().sort_index(),
    "porcentaje": (
        df_model["popular"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

target_summary.index = ["No popular (0)", "Popular (1)"]

target_summary

,cantidad,porcentaje
No popular (0),68530,76.37
Popular (1),21200,23.63


In [11]:
# Indicador de ausencia de información rítmica

df_model["rhythm_missing"] = (
    (df_model["tempo"] == 0)
    | (df_model["time_signature"] == 0)
).astype(int)

rhythm_summary = pd.DataFrame({
    "cantidad": df_model["rhythm_missing"].value_counts().sort_index(),
    "porcentaje": (
        df_model["rhythm_missing"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

rhythm_summary.index = [
    "Ritmo disponible (0)",
    "Ritmo no disponible (1)"
]

rhythm_summary

,cantidad,porcentaje
Ritmo disponible (0),89568,99.82
Ritmo no disponible (1),162,0.18


In [12]:
# Interacción entre energía y tempo

df_model["energy_tempo"] = (
    df_model["energy"] * df_model["tempo"]
)

df_model[
    ["energy", "tempo", "energy_tempo"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
energy,89730.0,0.634467,0.256610,0.0,0.457000,0.676500,0.853000,1.000
tempo,89730.0,122.058996,30.118657,0.0,99.260250,122.013000,140.077750,243.372
energy_tempo,89730.0,79.441037,40.242068,0.0,49.758736,78.411687,107.246103,212.246


In [13]:
# Indicador de pistas de larga duración

df_model["is_long_track"] = (
    df_model["duration_ms"] >= 600_000
).astype(int)

long_track_summary = pd.DataFrame({
    "cantidad": df_model["is_long_track"].value_counts().sort_index(),
    "porcentaje": (
        df_model["is_long_track"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

long_track_summary.index = [
    "Duración menor a 10 min (0)",
    "Duración de 10 min o más (1)"
]

long_track_summary

,cantidad,porcentaje
Duración menor a 10 min (0),89180,99.39
Duración de 10 min o más (1),550,0.61


In [14]:
# Reemplazar ceros no válidos por valores faltantes técnicos

df_model["tempo"] = df_model["tempo"].replace(0, np.nan)
df_model["time_signature"] = df_model["time_signature"].replace(0, np.nan)

print("Valores faltantes en tempo:", df_model["tempo"].isna().sum())
print("Valores faltantes en time_signature:", df_model["time_signature"].isna().sum())

Valores faltantes en tempo: 157
Valores faltantes en time_signature: 162


In [15]:
# Recalcular interacción energía × tempo después de marcar tempos faltantes

df_model["energy_tempo"] = (
    df_model["energy"] * df_model["tempo"]
)

print("Valores faltantes en energy_tempo:", df_model["energy_tempo"].isna().sum())

df_model[
    ["energy", "tempo", "energy_tempo", "rhythm_missing"]
].head()

Valores faltantes en energy_tempo: 157


,energy,tempo,energy_tempo,rhythm_missing
0,0.374,104.042,38.911708,0
1,0.516,178.174,91.937784,0
2,0.560,119.997,67.198320,0
3,0.770,161.721,124.525170,0
4,0.431,129.971,56.017501,0


In [16]:
print("Shape final temporal:", df_model.shape)

print("\nColumnas:")
print(df_model.columns.tolist())

print("\nValores nulos:")
display(df_model.isna().sum()[df_model.isna().sum() > 0])

Shape final temporal: (89730, 24)

Columnas:
['track_id', 'artists', 'album_name', 'track_name', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'popularity', 'genres', 'popular', 'rhythm_missing', 'energy_tempo', 'is_long_track']

Valores nulos:


tempo             157
time_signature    162
energy_tempo      157
dtype: int64

In [17]:
# Preparar y exportar dataset limpio para modelado

df_export = df_model.copy()

# Convertir listas de géneros a texto separado por |
df_export["genres"] = df_export["genres"].apply(
    lambda genres: "|".join(genres)
)

output_path = "../data/processed/spotify_model_dataset.csv"

df_export.to_csv(
    output_path,
    index=False
)

print("Archivo guardado en:", output_path)
print("Shape exportado:", df_export.shape)

df_export[
    [
        "track_id",
        "track_name",
        "genres",
        "popularity",
        "popular",
        "rhythm_missing",
        "energy_tempo",
        "is_long_track"
    ]
].head()

Archivo guardado en: ../data/processed/spotify_model_dataset.csv
Shape exportado: (89730, 24)


,track_id,track_name,genres,popularity,popular,rhythm_missing,energy_tempo,is_long_track
0,0000vdREvCVMxbQTkS888c,Lolly,german,44.0,0,0,38.911708,0
1,000CC8EParg64OmTxVnZ0p,It's All Coming Back To Me Now (Glee Cast Vers...,club,47.0,0,0,91.937784,0
2,000Iz0K615UepwSJ5z2RE5,Böxig Leise - Pig & Dan Remix,minimal-techno,22.0,0,0,67.198320,0
3,000RDCYioLteXcutOjeweY,Teeje Week,hip-hop,62.0,1,0,124.525170,0
4,000qpdoc97IMTBvF8gwcpy,Tief,minimal-techno,19.0,0,0,56.017501,0


In [18]:
# Definir variables para modelado y variables solo de referencia

reference_columns = [
    "track_id",
    "artists",
    "album_name",
    "track_name",
    "popularity"   # No puede entrar al modelo: se usa para crear el target
]

feature_columns = [
    "duration_ms",
    "explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
    "genres",
    "rhythm_missing",
    "energy_tempo",
    "is_long_track"
]

target_column = "popular"

print("Número de features:", len(feature_columns))
print("\nFeatures seleccionadas:")
print(feature_columns)

print("\n¿Hay columnas faltantes?")
print(set(feature_columns + [target_column]) - set(df_model.columns))

Número de features: 18

Features seleccionadas:
['duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'genres', 'rhythm_missing', 'energy_tempo', 'is_long_track']

¿Hay columnas faltantes?
set()


In [19]:
from sklearn.model_selection import train_test_split

# X contiene solo features; y contiene el target
X = df_model[feature_columns].copy()
y = df_model[target_column].copy()

# Referencias para trazabilidad, no entran al modelo
reference_data = df_model[reference_columns].copy()

# 15% reservado para evaluación final
X_train_val, X_test, y_train_val, y_test, ref_train_val, ref_test = train_test_split(
    X,
    y,
    reference_data,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Del 85% restante, separar validación para dejar aprox. 70/15/15 total
X_train, X_val, y_train, y_val, ref_train, ref_val = train_test_split(
    X_train_val,
    y_train_val,
    ref_train_val,
    test_size=0.1765,
    random_state=42,
    stratify=y_train_val
)

print("Train:", X_train.shape, "| Popular:", round(y_train.mean() * 100, 2), "%")
print("Validation:", X_val.shape, "| Popular:", round(y_val.mean() * 100, 2), "%")
print("Test:", X_test.shape, "| Popular:", round(y_test.mean() * 100, 2), "%")

Train: (62808, 18) | Popular: 23.63 %
Validation: (13462, 18) | Popular: 23.63 %
Test: (13460, 18) | Popular: 23.63 %


## Separación train, validation y test

El dataset de modelado se dividió usando estratificación sobre la variable objetivo `popular`.

| Conjunto | Registros | Propósito |
|---|---:|---|
| Train | 62.808 | Entrenamiento de modelos y ajuste inicial |
| Validation | 13.462 | Comparación de modelos, selección de umbral y tuning |
| Test | 13.460 | Evaluación final única del modelo seleccionado |

La proporción de canciones populares se mantuvo en 23,63% en los tres conjuntos.

El conjunto de prueba quedará reservado. No se utilizará para escoger features, ajustar hiperparámetros ni comparar modelos durante el desarrollo.